### Load Bronze data

In [0]:
silver_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/startup_india_funding/bronze"
)

### Remove Duplicates

In [0]:
silver_df = silver_df.dropDuplicates()

### Remove Completely Empty Rows

In [0]:
silver_df = silver_df.na.drop(how="all")

### Clean the Investment Amount

In [0]:
from pyspark.sql.functions import regexp_replace, col, trim

Now clean the amount column:

In [0]:
silver_df = silver_df.withColumn(
    "InvestmentAmount_USD",
    regexp_replace(trim(col("InvestmentAmount_USD")), ",", "")
)

Replace invalid values:

In [0]:
from pyspark.sql.functions import when

silver_df = silver_df.withColumn(
    "InvestmentAmount_USD",
    when(
        col("InvestmentAmount_USD").isin("", "NA", "N/A", "Unknown"),
        None
    ).otherwise(col("InvestmentAmount_USD"))
)

Convert to numeric:

In [0]:
silver_df = silver_df.withColumn(
    "InvestmentAmount_USD",
    col("InvestmentAmount_USD").cast("double")
)

### Standardize Text Columns

In [0]:
from pyspark.sql.functions import initcap

silver_df = silver_df.withColumn("City", initcap(col("City")))
silver_df = silver_df.withColumn("Industry", initcap(col("Industry")))
silver_df = silver_df.withColumn("InvestmentType", initcap(col("InvestmentType")))

### Convert Date

In [0]:
from pyspark.sql.functions import to_date

silver_df = silver_df.withColumn(
    "Date",
    to_date(col("Date"), "dd/MM/yyyy")
)

### Verify

In [0]:
display(silver_df)

Startup,Industry,SubVertical,City,Investors,InvestmentType,InvestmentAmount_USD,Date
Housejoy,Edtech,K12,Mumbai,Lightspeed India,Seed,199000.0,2023-04-19
SmartMart_37,Edtech,Test Prep,Kolkata,"A91 Partners, Falcon Edge, SoftBank Vision Fund",Seed,72000.0,2022-08-09
QuantumDrive,Retail,Fashion,Bengaluru,"Sequoia Capital India, Y Combinator",Series D,5.412E7,2024-04-16
Housejoy,Retail,Fashion,Bengaluru,"Falcon Edge, SoftBank Vision Fund, Y Combinator",Seed,1600000.0,2022-05-27
Dailyhunt,Media,Content,Mumbai,"Bessemer Venture Partners, Kedaara Capital",Pre-seed,84000.0,2023-07-14
Oyo,Consumer Electronics,Audio,Mumbai,Kalaari Capital,Seed,66000.0,2023-01-05
Practo,Realestate,Home Services,Noida,Tiger Global,Angel,81000.0,2023-08-13
Oyo,Consumer Electronics,Smart Home,Bengaluru,"Accel, Info Edge",Seed,366000.0,2022-09-07
NeoSense,Agritech,Marketplace,Hyderabad,"Accel, Kedaara Capital",Series A,1.1321E7,2023-05-24
FarmFit,Agritech,Supply Chain,Bengaluru,Ventures India,Seed,174000.0,2025-04-20


In [0]:
silver_df.printSchema()

root
 |-- Startup: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- SubVertical: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Investors: string (nullable = true)
 |-- InvestmentType: string (nullable = true)
 |-- InvestmentAmount_USD: double (nullable = true)
 |-- Date: date (nullable = true)



### Save Silver

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/startup_india_funding/silver")